为什么一堆线性层、点积、归一化、门控，最后能组成一个会“读上下文、记顺序、做推理”的模型？
## 3.1 注意力机制
如果 Embedding 做的是“把 token 变成向量”，那 Attention 做的就是让每个位置，根据自己的需求，去上下文里动态检索信息。
也就是所谓的 **Scaled Dot-Product Attention**： 

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

- **Query（Q）**：我现在想找什么？
- **Key（K）**：我这里有什么标签，别人能不能通过我找到我？
- **Value（V）**：如果别人真的关注我，我具体把什么内容交给它？

### 3.1.1 最后一次
这是我最后一次去以下面的例子学习普通点积自注意力。假设句子是：
```
["我", "吃", "苹果"]
```
我们观察最后一个 token —— “苹果”。这时候它面对的歧义是：

- “苹果”可以是水果
- 也可以是 Apple 公司 / 手机

那模型怎么判断？它不会查字典，而是靠上下文做语义对齐（这是在训练中发生的！）。

| 组件 | 问题                | 在“苹果”例子里的直觉      |
| -- | ----------------- | ---------------- |
| Q  | 我现在需要什么信息？        | “我到底是食物还是电子产品？”  |
| K  | 我能提供什么线索？         | “吃”携带“进食动作”的标签   |
| V  | 如果你关注我，我真正给你什么内容？ | “吃”会把“食物相关语义”传过来 |

于是计算发生了：
1. “苹果”的 Query 去和全句每个词的 Key 做相似度匹配
2. 它发现自己和“吃”的 Key 很匹配
3. softmax 后，“吃”拿到一部分注意力权重
4. 最后输出向量变成 $ 0.6 \cdot V_{\text{苹果}} + 0.3 \cdot V_{\text{吃}} + \cdots $

### 3.1.2 形状流水线
```
输入隐藏状态 X
[B, S, d_model]

线性投影（三个全局共享的线性层权重）
Q = XW_Q  -> [B, S, d_k]
K = XW_K  -> [B, S, d_k]
V = XW_V  -> [B, S, d_v]

分数矩阵
Q @ K^T   -> [B, S, S]
[
  [我→我,  我→吃,  我→苹果],
  [吃→我,  吃→吃,  吃→苹果],
  [苹果→我, 苹果→吃, 苹果→苹果],
]


softmax
weights   -> [B, S, S]

加权求和
weights @ V -> [B, S, d_v]
```
比如说`我 → 吃 → 苹 → 果`，用前 3 个 token，预测第 4 个 token = 果。
所有 Token Embedding 都是随机初始化的，Wq、Wk、Wv 也是随机初始化，算出来的 Q、K 全是随机向量，`Q @ K^T` 注意力分数完全乱给。
可能 “苹” 的注意力全跑到 “我” 身上，根本不看 “吃”，预测下一个 token 的概率一塌糊涂。
然而模型发现只要让 “对预测有用的 token” 拥有更高的 QK 点积，交叉熵损失就会变小。
于是反向传播时梯度就会自动调整参数 Wq、Wk、Embedding，让下一次预测 “果” 的概率变高，损失变小。
在无数次迭代里，会试出一个最优策略：
想猜对 “果”，必须让 “苹” 重点关注 “吃”想做到这一点，最简单的方法就是：让 “苹” 的 Q 和 “吃” 的 K 点积更大。
于是梯度就会：
- 调整 苹 的 Embedding
- 调整 吃 的 Embedding
- 调整 Wq，让 苹 算出的 Q 更偏向 “食物语义”
- 调整 Wk，让 吃 算出的 K 更偏向 “动作语义”
最终：Q_苹・K_吃 变得很大

从此以后：
- 代表食物 / 物体的 token，Q 会长得很像
- 代表动作 / 吃的 token，K 会长得很像
- 它们的点积天然就大
- 注意力权重自动集中在有用的上下文
### 3.1.3 维度缩放：为什么要除以 $\boldsymbol{\sqrt{d_k}}$？
#### 第一层：点积的方差会随维度 $d_k$ 线性增长
我们先做最基础的数学假设：Query向量 $q$ 和 Key向量 $k$ 中的每一个元素，都是**相互独立、均值为0、方差为1**的随机变量（这也是Transformer权重初始化的默认目标）。

两个独立随机变量的点积计算为：
$$
q \cdot k = \sum_{i=1}^{d_k} q_i \cdot k_i
$$

我们可以严格推导它的均值和方差：
1. 单元素乘积的均值：$\mathbb{E}[q_i \cdot k_i] = \mathbb{E}[q_i] \cdot \mathbb{E}[k_i] = 0 \times 0 = 0$
2. 单元素乘积的方差：$\text{Var}(q_i \cdot k_i) = \mathbb{E}[(q_i k_i)^2] - (\mathbb{E}[q_i k_i])^2 = \mathbb{E}[q_i^2] \cdot \mathbb{E}[k_i^2] = 1 \times 1 = 1$
3. 求和后的总方差：$\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = d_k$

**点积结果的标准差为 $\sqrt{d_k}$，会随着 $d_k$ 的增大而线性增长**。
比如当 $d_k=128$ 时，点积的标准差就达到了 $\sqrt{128} \approx 11.3$，很容易出现绝对值极大的logits值。

#### 第二层：logits 绝对值过大，会让 softmax 进入梯度饱和区
我们先明确softmax的核心公式，对于注意力分数向量 $z = [z_1, z_2, ..., z_S]$，softmax输出为：
$$
\sigma_i = \frac{e^{z_i}}{\sum_{j=1}^S e^{z_j}}
$$

而softmax的雅可比矩阵（梯度传递的核心）为：
$$
\frac{\partial \sigma_i}{\partial z_j} = \sigma_i \left( \delta_{ij} - \sigma_j \right)
$$
其中 $\delta_{ij}$ 为克罗内克函数：当 $i=j$ 时取值为1，否则为0。

当某个logit $z_i$ 绝对值极大时，会出现两种极端情况：
1.  $z_i \gg$ 其他所有 $z_j$：$\sigma_i \approx 1$，其余 $\sigma_j \approx 0$，此时梯度 $\frac{\partial \sigma_i}{\partial z_j} \approx 0$
2.  $z_i \ll$ 其他所有 $z_j$：$\sigma_i \approx 0$，此时梯度 $\frac{\partial \sigma_i}{\partial z_j} \approx 0$

两种情况都会让softmax进入**饱和区**，核心结论是：
> **softmax 一旦饱和，反向传播的梯度会无限趋近于0**。

#### 第三层：梯度消失会直接导致 Q/K 投影参数更新停滞
注意力分数的核心来源是 $QK^\top$，而 $Q$ 和 $K$ 分别来自输入隐藏态与可学习参数 $W_Q$、$W_K$ 的线性投影：
$$
Q = X W_Q, \quad K = X W_K
$$

反向传播的链路是：
`交叉熵损失 ← softmax输出 ← 注意力logits ← QK^T ← W_Q/W_K ← 输入Embedding`

一旦softmax饱和，梯度在这一步就会被“拦腰截断”，回传到 $W_Q$、$W_K$ 的梯度会变得极小，参数更新速度会指数级变慢，模型根本学不到“该关注哪个token、该忽略哪个token”的核心能力。

整个失效链路可以总结为：
> **维度升高 → 点积方差爆炸 → logits绝对值过大 → softmax梯度饱和 → 反向传播梯度消失 → Q/K投影参数更新停滞**

#### 第四层：$\boldsymbol{\sqrt{d_k}}$ 是最朴素且有效的方差校准器
原始Transformer论文中，在softmax之前除以 $\sqrt{d_k}$，本质就是**把点积结果的方差拉回1，让logits始终保持在梯度健康的区间内**。

经过缩放后的点积为：
$$
\frac{q \cdot k}{\sqrt{d_k}}
$$
此时它的方差为：
$$
\text{Var}\left( \frac{q \cdot k}{\sqrt{d_k}} \right) = \frac{1}{d_k} \cdot \text{Var}(q \cdot k) = \frac{1}{d_k} \cdot d_k = 1
$$

这一步缩放有两个核心特性：
1.  **不改变相对相似度**：除以同一个常数，不会改变token之间的相似度排序，只缩放绝对值大小
2.  **稳定训练梯度**：把logits拉回softmax的线性区，避免梯度过早消失，让Q/K参数能持续学习

---

#### 拓展：既然 $\boldsymbol{\sqrt{d_k}}$ 已经能控尺度，为什么还要引入 QK-Norm？
##### 1. 固定缩放的局限性
工业界在超大规模、深层、长上下文模型的训练中发现，仅靠固定的 $\sqrt{d_k}$ 缩放，无法完全解决注意力logits漂移的问题，核心痛点有4个：
- **分布偏移问题**：训练过程中，Q/K向量的分布会随着参数更新发生偏移，不再严格满足“均值0、方差1”的初始假设，固定缩放无法适配动态变化的分布
- **长上下文场景失效**：当序列长度 $S$ 扩展到8K、32K甚至128K时，即使单头方差可控，大量token的点积叠加后依然会出现极值，导致softmax饱和
- **分组注意力（GQA/MQA）的适配问题**：现代大模型普遍使用GQA/MQA降低显存占用，Key/Value的头数远少于Query头数，固定缩放无法适配这种不对称的投影结构
- **深层模型的梯度累积问题**：几十上百层的Transformer堆叠后，每层的微小分布偏移会逐层放大，最终导致注意力logits彻底失控

##### 2. QK-Norm 的核心定义与工业界标准实现
QK-Norm（Query-Key Normalization）的核心思想是：**先对Q和K做逐头的归一化，再用可学习的温度系数替代固定的 $\sqrt{d_k}$ 缩放**，从根源上保证注意力logits的分布始终稳定。

主流工业界实现的标准公式如下：
1.  **逐头层归一化**：对每个注意力头的Q和K分别做LayerNorm，强制拉回均值0、方差1的标准分布
    $$
    \hat{Q} = \text{LayerNorm}(Q), \quad \hat{K} = \text{LayerNorm}(K)
    $$
    注：归一化仅在头维度（$d_k$ 维度）执行，不跨batch、序列和注意力头，保证每个头的分布独立可控。

2.  **可学习温度系数缩放**：用一个可学习的标量参数 $\tau$（初始值通常设为 $\sqrt{d_k}$）替代固定缩放，让模型自主调整注意力的“温度”
    $$
    \text{Attention}(Q,K,V) = \text{softmax}\left( \frac{\hat{Q} \hat{K}^\top}{\tau} \right) V
    $$

部分模型（如Qwen3、Gemma 3）还会做进阶优化：
- 对归一化后的Q/K做L2归一化，让点积等价于余弦相似度，取值范围严格限制在 $[-1, 1]$ 之间，彻底消除维度和序列长度带来的方差影响
- 给温度系数设置上下限，避免模型学出极端的温度值导致训练不稳定
- 对归一化后的Q/K加入极小的权重衰减，进一步约束分布波动

##### 3. QK-Norm 的核心优势
| 优势 | 说明 |
| :--- | :--- |
| 极致的训练稳定性 | 从根源上强制Q/K的分布稳定，即使在100+层、128K长上下文、超大batch的极端场景下，也能避免softmax饱和，彻底解决loss爆炸、训练崩溃的问题 |
| 更好的长上下文适配性 | 归一化后的点积天然等价于余弦相似度，不会随序列长度增长出现极值，在超长上下文场景下依然能保持精准的注意力聚焦 |
| 缓解灾难性遗忘 | 稳定的注意力分布能减少深层模型的参数震荡，提升模型的持续学习能力，在多轮微调、增量预训练场景下表现更优 |
| 更低的微调门槛 | 即使在小batch、低资源的微调场景下，也能保持训练稳定，不会出现梯度消失/爆炸的问题，大幅降低了大模型落地的工程门槛 |

##### 4. 主流大模型的实践落地
| 模型 | 应用说明 |
| :--- | :--- |
| Qwen3 | 技术报告中明确将QK-Norm作为基础架构的核心标配组件，搭配GQA、SwiGLU、RoPE、RMSNorm、Pre-Norm结构，在7B/14B/72B全尺寸模型上实现了训练稳定性和下游效果的双重提升 |
| Gemma 3 | Google在Gemma 3系列模型中引入了改进版的QK-Norm，针对长上下文场景做了L2归一化优化，在128K上下文长度下依然保持了极佳的注意力质量和长文本理解能力 |
| Llama 3.1/3.2 | Meta在长上下文版本的Llama模型中，引入了可选的QK-Norm分支，作为支撑1M超长上下文稳定训练的核心组件 |
| DeepSeek-V2/V3 | 深度求索的开源大模型中，将QK-Norm作为长上下文训练的标配，大幅降低了超长序列下的训练难度，同时提升了长文本任务的效果 |

##### 5. 补充说明
QK-Norm不是对 $\sqrt{d_k}$ 缩放的否定，而是**对它的升级和泛化**：
- 固定 $\sqrt{d_k}$ 是「先验的、静态的」缩放，只能适配权重初始化的理想分布
- QK-Norm是「自适应的、动态的」缩放，能适配训练全过程的分布变化

两者的核心目标始终完全一致：**控制注意力logits的尺度，避免softmax饱和，保证梯度健康传递，让模型能稳定学到有效的注意力模式**。

### 3.1.6 实现思路

In [ ]:
import torch
import math

def stable_softmax(x: torch.Tensor, dim: int=-1) -> torch.Tensor:
    # 先减去最大值来稳定数值
    x_max = x.max(dim=dim, keepdim=True).values
    x_exp = torch.exp(x - x_max)
    return x_exp / x_exp.sum(dim=dim, keepdim=True)

def scaled_dot_product_attention(
    q: torch.Tensor,        # [..., SeqLen_q也就是n, d_k]
    k: torch.Tensor,        # [..., SeqLen_k也就是m, d_k]
    v: torch.Tensor,        # [..., SeqLen_v也是m, d_v]
    mask: torch.Tensor | None = None,
):
    d_k = q.size(-1)        # d_k 是每个头的维度

    # 1. 打分
    scores = torch.einsum("...nd,...md->...nm", q, k)  # [Batch, SeqLen_q, SeqLen_k]
    # 2. mask
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))

    # 3. softmax得到注意力权重
    attn = stable_softmax(scores, dim=-1)  # [Batch, SeqLen_q, SeqLen_k]

    # 4. 加权求和得到输出
    output = torch.einsum("...nm,...md->...nd", attn, v)  # [Batch, SeqLen_q, d_v]
    return output, attn

In [ ]:
torch.manual_seed(0)

B, S, d = 1, 3, 2
q = torch.randn(B, S, d)
k = torch.randn(B, S, d)
v = torch.randn(B, S, d)

# 因果 mask：下三角为 True
mask = torch.tril(torch.ones(S, S, dtype=torch.bool))

out_no_mask, attn_no_mask = scaled_dot_product_attention(q, k, v, mask=None)
out_mask, attn_mask = scaled_dot_product_attention(q, k, v, mask=mask)

# print("No mask attention:\n", attn_no_mask[0])
# print("With causal mask:\n", attn_mask[0])

No mask attention:
 tensor([[0.6601, 0.1685, 0.1714],
        [0.0782, 0.4458, 0.4760],
        [0.0363, 0.6953, 0.2684]])
With causal mask:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.1493, 0.8507, 0.0000],
        [0.0363, 0.6953, 0.2684]])


## 3.2 因果多头自注意力
模型里实际跑的不是“单头、无约束”的注意力，而是 **因果的、多头的、自注意力**。
* **Self**：Q/K/V 都来自同一段输入
* **Causal**：当前位置不能看未来
* **Multi-Head**：不是只用一个视角看上下文，而是多个子空间并行看

### 3.2.1 为什么非得多头？
单头自注意力里，每个 token 只能输出一组加权后的 V 向量，相当于把所有语义信息硬揉在一个向量（**同一个表示子空间**）里，根本没法同时捕捉多种关联。
举个我们之前熟悉的例子：`我用苹果电脑吃苹果`
- 单头注意力里，两个「苹果」的 QK 相似度会完全混在一起
- 模型没法同时抓住「苹果→电脑」的电子产品关联，和「苹果→吃」的食物关联
- 最终只能输出一个「折中混合」的注意力权重，两种语义都抓不准
MHA 的的核心操作非常简单：
1.  把模型隐藏维度 `d_model` 平均拆成 `h` 个独立的头（head），每个头的维度 `d_k = d_model / h`
2.  给每个头分配**独立的、互不共享的 `W_Q/W_K/W_V` 投影矩阵**，每个头都能看到完整的输入隐藏态`[B, S, d_model]`，每个头都有一套完全独立、互不共享、初始值随机的W_Q/W_K/W_V投影矩阵。
3.  每个头**完全独立地算一遍自注意力**，每个头专门学一种语义模式
4.  最后把所有头的输出拼接起来，再用一个线性层投影回 `d_model` 维度
比如8头注意力里：
- 头1：专门学「主谓宾/定状补」语法结构关联
- 头2：专门学「代词→先行词」的指代消解
- 头3：专门学「实体→属性」的常识关联
- 头4：专门学跨句子的长距离依赖
- 头5：专门学否定/转折等逻辑关系
- ... 

> 这听起来人为的刻意设计不太可能，但是实际上多头一般都不会趋同而是分化。
> 每个头的W_Q/W_K/W_V都是独立随机初始化的，初始的线性变换视角完全不同。就像 8 个盲人摸象，每个人初始站的位置完全不一样，摸到的部位不一样，最终形成的对 “大象” 的认知，也绝对不可能完全一样。哪怕训练过程中，它们都在往「降低损失」的方向走，也只会从不同的起点，走到不同的「局部最优解」，每个最优解对应一种不同的语义模式，绝不会趋同。
>
> Transformer 原文也可视化了不同头的注意力分布，发现有的头专门捕捉相邻 token 的语法关联，有的头专门捕捉跨句子的长距离依赖，完全是训练自动形成的，没有任何人工干预。后续大量针对 BERT 的分析论文发现，BERT 的 12 层 ×12 头注意力中，有专门负责指代消解的头、专门负责句法分析的头、专门负责否定词逻辑的头，甚至有专门负责数字关联的头 —— 这些分工全是训练自动形成的。


### 3.2.2 形状流水线
```
输入
x: [B, S, D]

Q/K/V 投影
q, k, v: [B, S, D]

拆头
[B, S, D]
-> [B, S, H, d_head]
-> [B, H, S, d_head]

每个头独立算注意力
scores: [B, H, S, S]
attn:   [B, H, S, S]
out:    [B, H, S, d_head]

合并头
[B, H, S, d_head]
-> [B, S, H, d_head]
-> [B, S, D]

输出投影
[B, S, D]
```

### 3.2.3 长上下文场景的注意力优化方案
#### 3.2.3.1 SRAM内存墙
MHA虽然表达能力强，但在长上下文推理场景下，会遇到无法回避的**内存墙问题**，这也是我们优化注意力架构的核心动机：
1.  **K/V缓存的显存爆炸**：LLM推理时会用KV Cache缓存历史token的K/V向量，避免重复计算。MHA中每个Q头都对应独立的K/V头，序列长度S越大、头数H越多，KV Cache的显存占用就越高（显存占用≈2·B·S·D·层数）。
2.  **SRAM装不下，速度暴跌**：GPU的计算核心（Tensor Core）速度极快，但片上SRAM缓存容量极小（A100单SM的SRAM仅几十MB）。长上下文下，`[B, H, S, S]`的分数矩阵、KV Cache无法塞进SRAM，计算核心需要反复从速度慢一个数量级的HBM显存中搬运数据，90%以上的时间都在“等数据”，推理速度直接暴跌。
3.  **优化核心思路**：在尽量不损失模型效果的前提下，**减少KV头的数量，降低KV Cache的显存占用，让数据能塞进SRAM，减少HBM的反复读写**。  

#### 3.2.3.2 MQA
也即 Multi-Query Attention，多查询注意力。
**核心设计**：**Query保持H个独立头，Key和Value全局共享1个公共头**，所有Q头共用同一组K/V向量。
- 解决的痛点：极致压缩KV Cache的显存占用，把KV Cache的体积直接缩小到原来的1/H。
- 形状流水线核心变化：
  ```python
  # MHA：Q/K/V 均为 [B, H, S, d_head]
  # MQA：Q 保持 [B, H, S, d_head]，K/V 仅为 [B, 1, S, d_head]
  q: [B, H, S, d_head]
  k: [B, 1, S, d_head]  # 头维度=1，全局共享
  v: [B, 1, S, d_head]  # 头维度=1，全局共享

  # 分数计算时，K/V会自动广播到所有Q头，形状仍输出 [B, H, S, S]
  scores = q @ k.transpose(-2, -1) / math.sqrt(d_head)  # [B, H, S, S]
  ```
- **优缺点**：
  - 优势：显存占用最低、推理速度最快，完美适配长上下文场景，KV Cache读写压力极小。
  - 劣势：效果损失明显，相比MHA会出现精度下降，尤其在复杂推理、长文本理解任务上。
- **代表落地模型**：PaLM、Llama 2、StarCoder。
#### 3.2.3.3 GQA
也即 Grouped-Query Attention，分组查询注意力。
**核心设计**：MHA和MQA的折中方案——**把H个Q头分成G个组，每个组共享1个独立的K/V头**。
- 举个例子：H=32个Q头，分成G=8个组，每个组4个Q头，共用1个K/V头，总共有8个K/V头，既压缩了显存，又保留了效果。
- 形状流水线核心变化：
  ```python
  # GQA核心形状
  G = 8  # 分组数，必须满足 H % G == 0
  q: [B, H, S, d_head]          # Q头数保持H=32
  k: [B, G, S, d_head]          # KV头数=分组数G=8
  v: [B, G, S, d_head]          # KV头数=分组数G=8

  # 计算时，每个组内的Q头共享本组的K/V，最终输出形状仍为 [B, H, S, S]
  ```
- **优缺点**：
  - 优势：完美平衡效果与速度，显存占用和推理速度接近MQA，模型效果几乎与MHA持平，是当前工业界的绝对主流方案。
  - 无明显短板，仅需提前设计好分组数，保证H能被G整除。
- **代表落地模型**：Llama 3/3.1/3.2、Qwen全系列、DeepSeek、Gemini、Mistral。
#### 3.2.3.4 MLA
也即 Multi-head Latent Attention，多头隐层注意力。
**核心设计**：Llama 3.1 128K长上下文版本推出的进阶优化方案，在GQA的基础上进一步压缩KV的维度——**把K/V向量压缩到更低维度的隐空间中，推理时再还原，极致降低KV Cache的显存占用**。
- 核心优化：传统MHA/GQA中，K/V的维度和Q一致；MLA中，先把K/V压缩成低维的隐向量存入KV Cache，计算注意力时再通过线性层还原到目标维度，KV Cache的体积可再降低50%以上。
- **优缺点**：
  - 优势：在GQA的基础上进一步降低长上下文的显存占用，128K+超长上下文场景下效果和速度兼顾，是当前最先进的注意力架构之一。
  - 劣势：工程实现复杂度更高，需要配套的压缩/还原线性层设计。
- **代表落地模型**：Llama 3.1 长上下文版本、DeepSeek V2/V3、Qwen2 超长上下文版本。

> 另外**所有优化方案，仅改变KV的头数/维度，不改变最终输出形状**。MQA/GQA/MLA的最终输出形状仍为`[B, S, D]`，可无缝替换MHA，无需修改Transformer的其他结构。训练时使用什么注意力架构，推理时就必须用对应的架构，KV Cache的格式完全由注意力架构决定。
> 
> FlashAttention的核心是分块计算解决SRAM装不下的问题，而MQA/GQA/MLA是从架构上减少KV的数据量，两者是互补关系，工业界普遍是「GQA + FlashAttention」组合使用，实现长上下文推理的极致加速。
### 3.2.4 实现思路
> 没引入ROPE，不太完整的最小实现

In [ ]:
import sys
from pathlib import Path
# 把项目根目录加入 Python 路径
ROOT = Path("..").absolute()
sys.path.append(str(ROOT))

import torch
import torch.nn as nn
import import_ipynb
from transformer.transformer import TransformerLinear

class CausalMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int,
                max_seq_len: int, rope_theta: float,    # 我丢无默认值的要放前面
                device=None, dtype=None,
    ):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # 定义线性层
        self.q_proj = TransformerLinear(d_model, d_model, bias=False)
        self.k_proj = TransformerLinear(d_model, d_model, bias=False)
        self.v_proj = TransformerLinear(d_model, d_model, bias=False)
        self.out_proj = TransformerLinear(d_model, d_model)

        # RoPE
        self.rope = RotaryPositionalEmbedding(
            theta = rope_theta,
            d_head = self.d_k,
            max_seq_len = max_seq_len,
            device=device
        )

    def forward(self, x:torch.Tensor, token_positions: torch.Tensor = None):
        B, S, D = x.shape
        H, Dh = self.num_heads, self.d_k

        # view的作用是把最后一个维度 d_model 切分成 num_heads 个头，每个头的维度是 d_k
        # transpose的作用是把维度调整成 [Batch, num_heads, SeqLen, d_k]，方便后续计算
        # 维度变化是 [B, S, D] -> [B, S, H, Dh] -> [B, H, S, Dh]
        q = self.q_proj(x).view(B, S, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, H, Dh).transpose(1, 2)

        # 从decoder回来的时候加的
        if token_positions is not None:
            q = self.rope(q, token_positions)
            k = self.rope(k, token_positions)

        # tril函数生成一个下三角矩阵，表示因果 mask
        mask = torch.tril(torch.ones(S, S, dtype=torch.bool, device=x.device))

        output, attn = scaled_dot_product_attention(q, k, v, mask=mask)
        # 把多头的输出拼接回 [Batch, SeqLen, d_model]
        # contiguous的作用是确保内存连续，view才能正确工作
        # 维度变化是 [B, H, S, Dh] -> [B, S, H, Dh] -> [B, S, D]
        output = output.transpose(1, 2).contiguous().view(B, S, D)
        # 最后通过线性层得到最终输出
        output = self.out_proj(output)
        return output

: 

## 3.3 位置编码
前面我们说的已经很详细了，**纯 attention 本身对顺序并不敏感**。而语言是强顺序结构，所以必须补位置信息。
### 3.3.1 从相加到旋转
早期Transformer（如GPT-2、BERT）的位置编码方案是**绝对位置编码**：预定义一组位置向量，直接加到Token Embedding上输入模型。这种方案虽然能用，但存在三个天然硬伤：
1.  **位置信息是“外挂”的**：位置向量和语义向量简单相加，位置没有真正融入语义交互的核心逻辑。
2.  **相对位置依赖需要模型“硬学”**：模型需要从训练数据中自己摸索“第m个token和第n个token的相对距离是m-n”，不够优雅且数据效率低。
3.  **长上下文外推性差**：训练时只见过S长度的位置，推理时遇到S+1的位置，模型很难自然泛化。

RoPE（Rotary Position Embedding，旋转位置编码）的设计思路则完全不同，它的核心洞察是：
> **不再把位置信息直接加到输入上，而是直接把位置编码作用在注意力的核心——Query和Key向量上，通过“旋转”来编码位置。**

RoFormer论文明确指出：RoPE通过旋转矩阵编码**绝对位置**，同时在自注意力的点积计算中**自然体现相对位置依赖**，兼具序列长度灵活性、相对距离衰减等优良性质。这种设计让位置不再是“附加属性”，而是直接进入“谁该关注谁”的打分过程，是位置编码方案的一次范式升级。

### 3.3.2 它到底怎么转
RoPE的数学基础非常简洁：**把一个d维向量看成d/2个独立的二维平面，对每一对维度做一次独立的二维旋转**。
#### 第一步：二维旋转矩阵
对于任意一个二维向量 $[x, y]$，绕原点旋转角度 $\theta$ 后的新向量 $[x', y']$ 可以用旋转矩阵表示：
$$
R(\theta) = \begin{bmatrix}
\cos\theta & -\sin\theta \\
\sin\theta & \cos\theta
\end{bmatrix}
$$
旋转操作的矩阵乘法形式为：
$$
\begin{bmatrix}
x' \\
y'
\end{bmatrix} = R(\theta) \cdot \begin{bmatrix}
x \\
y
\end{bmatrix}
$$

#### 第二步：推广到d维向量
对于一个d维的Query或Key向量（d必须是偶数，这也是RoPE的一个小约束），我们把它拆成d/2个独立的二维平面对：
- 第0维和第1维组成第一个平面
- 第2维和第3维组成第二个平面
- ...
- 第2k维和第2k+1维组成第k+1个平面

对于位置为 $i$ 的token，它的第 $k$ 个平面对应的旋转角度为 $\theta_{i,k}$，我们对每一对维度独立应用旋转矩阵：
$$
\begin{bmatrix}
x_{2k}' \\
x_{2k+1}'
\end{bmatrix} = R(\theta_{i,k}) \cdot \begin{bmatrix}
x_{2k} \\
x_{2k+1}
\end{bmatrix}
$$
其中 $x_{2k}, x_{2k+1}$ 是旋转前的向量元素，$x_{2k}', x_{2k+1}'$ 是旋转后的元素。

#### 第三步：整体的分块对角旋转矩阵
把d/2个独立的二维旋转矩阵拼在一起，就得到了d维的整体旋转矩阵 $R_d(\theta_i)$，它是一个分块对角矩阵：
$$
R_d(\theta_i) = \begin{bmatrix}
R(\theta_{i,0}) & 0 & \dots & 0 \\
0 & R(\theta_{i,1}) & \dots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \dots & R(\theta_{i,d/2-1})
\end{bmatrix}
$$
最终，位置为 $i$ 的token的Query向量旋转后为：
$$
\tilde{q}_i = R_d(\theta_i) \cdot q_i
$$
Key向量的旋转操作完全一致：$\tilde{k}_i = R_d(\theta_i) \cdot k_i$。

### 3.3.3 为什么角度要分频率
RoPE最精妙的设计之一就是**不同维度对的旋转速度（频率）完全不同**，这让模型能同时感知不同尺度的位置信息。
第 $k$ 个维度对、位置为 $i$ 的旋转角度定义为：
$$
\theta_{i,k} = \frac{i}{\Theta^{2k/d}}
$$

| 符号 | 含义 | 取值规则 |
|------|----------|----------|
| $i$ | token的**绝对位置** | 从0开始数，比如句子「我/吃/苹果」，「我」的$i=0$，「吃」$i=1$，「苹果」$i=2$ |
| $k$ | **维度对的序号** | 我们把d维的Q/K向量拆成$d/2$个二维平面对，$k$就是这一对的编号，从0开始，最大到$(d/2)-1$ |
| $d$ | 单个注意力头里，Q/K向量的总维度 | 必须是偶数，比如常见的$d=64/128$ |
| $\Theta$（Theta） | 频率基，超参数 | 原始论文默认10000，长上下文模型会调大到10万、100万甚至1000万 |
| $\theta_{i,k}$ | 核心：**位置i的第k个维度对，要旋转的角度** | 单位是弧度，我们平时说的360°=2π弧度 |

**它本质就是：旋转角度 = 绝对位置 × 该维度对的转速（角频率）**
$$
\theta_{i,k} = i \times \underbrace{\frac{1}{\Theta^{2k/d}}}_{\omega_k}
$$
其中 $\omega_k$ 就是**第k个维度对的「角频率/转速」**。

这个逻辑和我们现实中的钟表完全一模一样：
- $i$ = 时间（秒）
- $\omega_k$ = 指针的转速（每秒转多少弧度）
- $\theta_{i,k}$ = 到了第i秒，这个指针一共转了多少角度
**位置越靠后，转的角度必须越大**。
- 位置0（$i=0$）：角度必须是0，不旋转，作为基准
- 位置1（$i=1$）：转一点
- 位置100（$i=100$）：转更多
只有这样，不同位置的token，旋转后的向量才会不一样，模型才能区分「第1个token和第100个token的位置不同」。

如果所有维度对的转速都一样，那所有平面对都转了相同的角度，等于把整个向量整体旋转，和没拆成对没有任何区别，完全浪费了多维度的空间。

只有让不同$k$的转速不一样，我们才能实现**多尺度的位置感知**：
- 转速快的维度对：对「相邻位置的微小变化」极其敏感，抓局部词序、语法结构
- 转速慢的维度对：对「长距离的位置差异」才敏感，抓跨段落的指代、长程依赖

#### 3.3.3.1 $2k/d$：归一化，保证不同维度的模型频率范围一致
$2k/d$的作用是把$k$的取值，**归一化到0~1之间**：
- 当$k=0$（第一个维度对）：$2*0/d=0$
- 当$k=d/2-1$（最后一个维度对）：$2*(d/2-1)/d ≈ 1$

不管你的向量维度$d$是64还是4096，所有维度对的转速都会落在**1 ~ 1/Θ**这个固定区间里，不会因为维度变大，转速变得过快或过慢。
#### 3.3.3.2 指数形式：实现对数均匀的频率分布，覆盖多尺度位置
指数衰减的转速，会让频率呈**对数均匀分布**，这是信号处理里做多尺度分析的标准做法，和我们人耳能听到20Hz~20000Hz的声音是一个逻辑——用对数尺度覆盖多个数量级的范围。
不同频率的旋转项叠加后，距离越远，越容易相消，注意力分数的包络线会往下掉。大多数情况下，近处的词往往比远处的词更相关，这很符合语言直觉。

### 3.3.4 RoPE 为什么天然适合 Attention
自注意力的计算逻辑在于，**两个位置旋转后的点积，会自然变成仅关于相对位置 $m-n$ 的函数**。
旋转后的Query和旋转后的Key的点积 $\tilde{q}_m^\top \tilde{k}_n$。

对于任意两个位置 $m$ 和 $n$，我们可以证明：
$$
\tilde{q}_m^\top \tilde{k}_n = (R_d(\theta_m) q_m)^\top (R_d(\theta_n) k_n) = q_m^\top R_d(\theta_m)^\top R_d(\theta_n) k_n
$$
由于旋转矩阵是正交矩阵（$R^\top R = I$），且旋转矩阵的乘积等价于角度相加的旋转，我们可以得到：
$$
R_d(\theta_m)^\top R_d(\theta_n) = R_d(\theta_n - \theta_m)
$$
而 $\theta_n - \theta_m$ 仅和相对位置 $n-m$ 有关，和绝对位置 $m,n$ 无关。因此：
$$
\tilde{q}_m^\top \tilde{k}_n = f(q_m, k_n, m-n)
$$
这意味着：**模型在计算注意力分数时，天然就对“相对距离”敏感，不需要额外学习相对位置关系**。

相比之下，绝对位置编码是“先把绝对位置加到Embedding上，再让模型自己从数据中摸索相对关系”，而RoPE直接把相对位置编码进了注意力分数的计算中，这是一种更优雅、数据效率更高的设计。

### 3.3.5 实现思路
**奇偶位拆分 + 逐元素cos/sin组合**

假设我们有一个4维的Query向量 $q = [x_0, x_1, x_2, x_3]$，位置为 $i$：

1.  **奇偶位拆分**：
    - 偶数位：$q_{\text{even}} = [x_0, x_2]$
    - 奇数位：$q_{\text{odd}} = [x_1, x_3]$

2.  **预计算旋转角度的cos和sin值**：
    对于位置 $i$ 和每个维度对 $k$，预计算：
    $$
    \cos_k = \cos(\theta_{i,k}), \quad \sin_k = \sin(\theta_{i,k})
    $$
    对于4维向量，我们得到 $\cos = [\cos_0, \cos_1]$，$\sin = [\sin_0, \sin_1]$。

3.  **逐元素组合（核心旋转操作）**：
    旋转后的偶数位和奇数位分别为：
    $$
    q_{\text{even}}' = q_{\text{even}} \odot \cos - q_{\text{odd}} \odot \sin
    $$
    $$
    q_{\text{odd}}' = q_{\text{even}} \odot \sin + q_{\text{odd}} \odot \cos
    $$
    其中 $\odot$ 表示逐元素乘法（Hadamard乘积）。
    假设我们的输入张量是多头拆分后的Query：
    - `q.shape = [B, H, S, D]` （Batch, 头数, 序列长度, 维度）

    我们预计算的cos和sin张量形状是：
    - `cos.shape = [S, D]` （序列长度, 维度）
    - `sin.shape = [S, D]`

    PyTorch的广播机制会自动把 `[S, D]` 补成 `[1, 1, S, D]`，再广播到 `[B, H, S, D]`，和q的形状完全对齐。因此我们可以直接写：
    ```python
    # 正确的代码：形状自动对齐
    q_rot = (q * cos) + (torch.roll(q, shifts=1, dims=-1) * sin)
    ```
    如果不小心把张量的维度顺序写错了，比如写成 `[B, S, H, D]`，那 `[S, D]` 的cos/sin就会和中间的 `H` 维度“打架”，广播会出错。这时候必须手动调整维度顺序或用 `unsqueeze` 显式扩展维度：
    ```python
    # 错误的形状顺序：[B, S, H, D]
    # 需要手动调整
    cos = cos.unsqueeze(0).unsqueeze(2)  # [S, D] -> [1, S, 1, D]
    q_rot = (q * cos) + (torch.roll(q, shifts=1, dims=-1) * sin)
    ```
    确保cos/sin的序列维S和特征维D，对齐到输入张量的**最后两位**，广播就能自然工作。

4.  **交错拼接回d维向量**：
    把 $q_{\text{even}}'$ 和 $q_{\text{odd}}'$ 交错塞回去，得到最终的旋转后向量：
    $$
    \tilde{q} = [q_{\text{even},0}', q_{\text{odd},0}', q_{\text{even},1}', q_{\text{odd},1}']
    $$


In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, theta: float, d_head: int, max_seq_len: int, device=None):
        super().__init__()
        assert d_head % 2 == 0, "d_head must be even for rotary embeddings"
        self.d_head = d_head

        # 频率
        idx = torch.arange(0, d_head, 2).float() / d_head # [0, 2, 4, ..., d_head-2] / d_head
                                                          # 生成从0开始，步长为2，到d_head（不包含）的序列
        freqs = 1.0 / (theta ** idx)

        # 生成位置向量
        positions = torch.arange(max_seq_len).float()

        # 外积
        # 输入 a 是长度为 M 的向量，b 是长度为 N 的向量
        # 输出是 [M, N] 的矩阵，第 i 行第 j 列 = a[i] * b[j]
        angles = torch.outer(positions, freqs)  # [max_seq_len, d_head//2]

        # 缓存cos和sin
        self.register_buffer("cos_cached", angles.cos(), persistent=False)  # [max_seq_len, d_head//2]
        self.register_buffer("sin_cached", angles.sin(), persistent=False)  # [max_seq_len, d_head//2]
        # 对于angles里的每个数单独算cos和sin

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        # x: [B, H, S, D] or [B, S, D]
        # 这里__init__里预先计算好了cos和sin的值，forward里直接根据token_positions索引就可
        cos = self.cos_cached[token_positions]
        sin = self.sin_cached[token_positions]

        # 处理维度
        if x.ndim == 4 and cos.ndim == 3:
        # 若X是[B, H, S, D], 需要在cos和sin前面加一个维度以匹配
            cos = cos.unsqueeze(1)  # [B, 1, S, D//2]
            sin = sin.unsqueeze(1)  # [B, 1, S, D//2]
        cos = cos.to(x.dtype)
        sin = sin.to(x.dtype)

        # 奇偶位拆分并旋转
        x_even = x[..., 0::2]       # 取偶数位
        x_odd = x[..., 1::2]        # 取奇数位

        out = torch.empty_like(x)
        out[..., 0::2] = x_even * cos - x_odd * sin
        out[..., 1::2] = x_even * sin + x_odd * cos
        return out

果然这里还是举一个例子吧。给定输入如下：
- 序列长度 `seq_len = S = 8`：一句话有8个token，位置编号从 `0~7`
- 单注意力头维度 `d_head = D = 8`：每个token的Q/K向量都是8维（主流模型常用64/128，比如Llama 3的d_head=128，这里用8是为了手算清晰）
- 频率基 `theta = 10000`

| 矩阵 | 形状 | 核心含义 |
|------|------|----------|
| 原始Q矩阵 | `[S, D] = [8, 8]` | **Q的第i行 `q_i` = 第i个token的Query向量**，代表「第i个token想要检索什么信息」 |
| 原始K矩阵 | `[S, D] = [8, 8]` | **K的第j行 `k_j` = 第j个token的Key向量**，代表「第j个token拥有什么信息」 |
| 注意力分数矩阵 | `[S, S] = [8, 8]` | 第i行第j列的数值 = `q_i · k_j`（两个向量的点积），代表「第i个token对第j个token的关注程度」 |

> 注意力的核心是 `q_i · k_j`，模型「该关注谁、不该关注谁」的所有决策，都来自这个点积。**只有把位置信息直接注入到Q和K里，才能让「位置关系」直接决定注意力打分，而不是靠模型硬学**。
> 如果我们不给Q/K加任何位置信息，会发生什么？
> 举个极端例子：序列是 `AAAAAAAA`（8个完全相同的token），那么：
> - 每个token的语义向量完全一样 → 原始Q矩阵的8行全相同，原始K矩阵的8行全相同
> - 注意力分数矩阵的**所有元素都完全相等**（`q_i · k_j` 对任意i,j都一样）
> - 模型完全分不清：「第1个token和第2个token（距离1）」和「第1个token和第8个token（距离7）」有任何区别，连语序都无法识别。
> 当然一般在分词的时候就会寄

RoPE不改变Q/K的语义信息，只做一件事：
> 给**每个位置i的q_i和k_i**，做一次**和位置i强绑定的线性旋转**。旋转角度只和「位置i」、「维度编号」有关，和语义无关。
1.  哪怕两个token的语义完全相同，只要位置不同，旋转后的q/k向量就不同，注意力分数自然带上位置信息；
2.  **数学上严格保证**：旋转后的 `q_i · k_j`，只和「语义相似度」+「相对位置 `i-j`」有关，和绝对位置i、j无关。
$$
\tilde{q}_i^\top \tilde{k}_j = f(q_i, k_j, i-j)
$$

### 3.3.6 手动推导
RoPE的旋转角度公式为：
$$
\theta_{i,k} = \frac{i}{\Theta^{2k/d}}
$$
其中：
- `i`：token的位置编号（0~7）
- `k`：二维平面对的编号（d_head=8，拆成4个二维平面对，k=0,1,2,3）
- `d`：d_head=8
- `Θ`：theta=10000

我们先算**每个平面对的频率 `freqs_k = 1 / Θ^{2k/d}`**，这个频率决定了该平面对的旋转速度：频率越高，转得越快
| 平面对编号k | 2k/d | Θ^{2k/d} | freqs_k = 1/Θ^{2k/d} | 转速特性 |
|-------------|------|----------|------------------------|----------|
| k=0（第0、1维） | 0/8=0 | 10000^0=1 | 1.0 | 最高频（秒针），位置每+1，角度+1弧度 |
| k=1（第2、3维） | 2/8=0.25 | 10000^0.25=10 | 0.1 | 中频（分针），位置每+1，角度+0.1弧度 |
| k=2（第4、5维） | 4/8=0.5 | 10000^0.5=100 | 0.01 | 低频（时针），位置每+1，角度+0.01弧度 |
| k=3（第6、7维） | 6/8=0.75 | 10000^0.75=1000 | 0.001 | 极低频，位置每+1，角度+0.001弧度 |

最终得到频率向量：
$$
\text{freqs} = [1.0, 0.1, 0.01, 0.001] \quad \text{shape} = [d/2] = [4]
$$

角度 = 位置i × 频率freqs，我们用**外积**一次性算出所有位置的角度：
$$
\text{angles} = \text{positions} \otimes \text{freqs}
$$
其中 `positions = [0,1,2,3,4,5,6,7]`（8个位置）。

| 位置i | k=0 | k=1 | k=2 | k=3 |
|-------|-----|-----|-----|-----|
| 0 | 0×1.0=0.0 | 0×0.1=0.0 | 0×0.01=0.0 | 0×0.001=0.0 |
| 1 | 1×1.0=1.0 | 1×0.1=0.1 | 1×0.01=0.01 | 1×0.001=0.001 |
| 2 | 2×1.0=2.0 | 2×0.1=0.2 | 2×0.01=0.02 | 2×0.001=0.002 |
| 3 | 3×1.0=3.0 | 3×0.1=0.3 | 3×0.01=0.03 | 3×0.001=0.003 |
| 4 | 4×1.0=4.0 | 4×0.1=0.4 | 4×0.01=0.04 | 4×0.001=0.004 |
| 5 | 5×1.0=5.0 | 5×0.1=0.5 | 5×0.01=0.05 | 5×0.001=0.005 |
| 6 | 6×1.0=6.0 | 6×0.1=0.6 | 6×0.01=0.06 | 6×0.001=0.006 |
| 7 | 7×1.0=7.0 | 7×0.1=0.7 | 7×0.01=0.07 | 7×0.001=0.007 |

- 高频维度（k=0）：位置从0到7，角度从0涨到7弧度（转了1圈多），变化极快，负责感知局部位置变化；
- 极低频维度（k=3）：位置从0到7，角度只涨了0.007弧度，几乎没动，负责保留全局长距离位置感。

接下来对角度矩阵里的每个值，分别预计算cos和sin，得到两个和angles形状完全相同的矩阵：
- `cos_cached.shape = [8,4]`：每个位置、每个平面对的cos值
- `sin_cached.shape = [8,4]`：每个位置、每个平面对的sin值

| 位置i | cos(k=0) | sin(k=0) | cos(k=1) | sin(k=1) | cos(k=2) | sin(k=2) | cos(k=3) | sin(k=3) |
|-------|----------|----------|----------|----------|----------|----------|----------|----------|
| 0 | 1.0000 | 0.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 | 1.0000 | 0.0000 |
| 1 | 0.5403 | 0.8415 | 0.9950 | 0.0998 | 0.99995 | 0.0100 | 1.0000 | 0.0010 |
| 2 | -0.4161 | 0.9093 | 0.9801 | 0.1987 | 0.9998 | 0.0200 | 1.0000 | 0.0020 |
| 3 | -0.9899 | 0.1411 | 0.9553 | 0.2955 | 0.99955 | 0.0300 | 1.0000 | 0.0030 |
| 7 | 0.7539 | -0.6570 | 0.7648 | 0.6442 | 0.99755 | 0.0700 | 1.0000 | 0.0070 |

为了**最大化对比效果**，我们用最极端的设定：
> 8个token的语义完全相同（比如序列是8个A），所以**原始Q矩阵的8行完全相同，原始K矩阵的8行也完全相同**。

这样，不用RoPE的话，注意力分数全是一样的，用了RoPE之后，位置信息的效果会被无限放大。

我们设定原始Q/K的每一行都是：
$$
q_i = k_j = [1, 0, 1, 0, 1, 0, 1, 0] \quad \forall i,j \in 0\sim7
$$
- 形状：Q = K = `[8, 8]`，8行全是上面的向量
- 不用RoPE的话，任意两个向量的点积：`1×1 + 0×0 + 1×1 + 0×0 + 1×1 + 0×0 + 1×1 + 0×0 = 4`
- 不用RoPE的注意力分数矩阵：**所有元素都是4**，完全没有位置差异。

进入旋转阶段，我们先对任意位置i的8维向量x，我们先拆成4个二维平面对：
$$
x = [x_0,x_1,x_2,x_3,x_4,x_5,x_6,x_7] \implies \text{even}=[x_0,x_2,x_4,x_6], \text{odd}=[x_1,x_3,x_5,x_7]
$$
旋转后的向量为：
$$
\begin{cases}
\text{out}_{\text{even}} = \text{even} \odot \cos_i - \text{odd} \odot \sin_i \\
\text{out}_{\text{odd}} = \text{even} \odot \sin_i + \text{odd} \odot \cos_i
\end{cases}
$$

我们算3个核心位置的旋转后q向量（k向量和q完全相同）：

1. 位置0（i=0）
cos=[1,1,1,1], sin=[0,0,0,0]，旋转后和原向量完全一致：
$$
\tilde{q}_0 = [1, 0, 1, 0, 1, 0, 1, 0]
$$

2. 位置1（i=1）
cos=[0.5403, 0.9950, 0.99995, 1.0], sin=[0.8415, 0.0998, 0.0100, 0.0010]
- even = [1,1,1,1], odd = [0,0,0,0]
- out_even = 1×0.5403 - 0×0.8415, 1×0.9950 - 0×0.0998, ... → [0.5403, 0.9950, 0.99995, 1.0]
- out_odd = 1×0.8415 + 0×0.5403, 1×0.0998 + 0×0.9950, ... → [0.8415, 0.0998, 0.0100, 0.0010]
- 交错拼接后：
$$
\tilde{q}_1 = [0.5403, 0.8415, 0.9950, 0.0998, 0.99995, 0.0100, 1.0, 0.0010]
$$

3. 位置2（i=2）
cos=[-0.4161, 0.9801, 0.9998, 1.0], sin=[0.9093, 0.1987, 0.0200, 0.0020]
同理计算得：
$$
\tilde{q}_2 = [-0.4161, 0.9093, 0.9801, 0.1987, 0.9998, 0.0200, 1.0, 0.0020]
$$

我们现在计算**用了RoPE之后的注意力分数矩阵**，也就是 $\tilde{Q} @ \tilde{K}^\top$，核心看两个关键性质：
#### 性质1：相对距离相同，注意力分数几乎完全相同
我们算三组相对距离=1的点积：
1.  位置1的q · 位置2的k（相对距离2-1=1）：
    $$
    \tilde{q}_1 \cdot \tilde{k}_2 ≈ 0.5403×(-0.4161) + 0.8415×0.9093 + 0.9950×0.9801 + 0.0998×0.1987 + ... ≈ 2.97
    $$
2.  位置2的q · 位置3的k（相对距离3-2=1）：
    $$
    \tilde{q}_2 \cdot \tilde{k}_3 ≈ 2.97
    $$
3.  位置0的q · 位置1的k（相对距离1-0=1）：
    $$
    \tilde{q}_0 \cdot \tilde{k}_1 ≈ 2.97
    $$

**结论**：只要相对距离相同，不管绝对位置在哪，注意力分数完全一致！完美验证了RoPE的核心承诺。

#### 性质2：相对距离越远，注意力分数自然衰减
我们算位置0的q，对不同位置的k的点积（相对距离从0到7）：
| 相对距离 | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
|----------|---|---|---|---|---|---|---|---|
| 注意力分数 | 4.0 | 2.97 | 1.94 | 0.95 | 0.02 | -0.85 | -1.59 | -2.16 |

**结论**：距离越远，分数越低，完全符合语言的直觉——近处的token更相关，远处的token更不相关。这个衰减是数学上天然保证的，不需要模型学习！

| 场景 | 注意力分数特点 | 模型能力 |
|------|----------------|----------|
| 不用RoPE | 所有位置的分数全是4.0，完全相同 | 完全分不清语序、远近，连“我吃苹果”和“苹果吃我”都分不清 |
| 用了RoPE | 相对距离相同，分数相同；距离越远，分数越低 | 天然懂语序、懂位置远近，不需要从数据里硬学位置关系 |



## 3.4 前馈网络：SwiGLU
我很喜欢把 Transformer Block 总结成六个字：**先交流，再内化。**
- Attention 是交流。
- FFN 是内化。
前者让 token 间互相通信，后者让每个 token 自己在本地做非线性加工。
如果没有 FFN，模型就会有点像“只会搬运上下文，不太会在本位置做复杂变换”的系统。
Attention 做完之后，每个位置已经拿到一份上下文摘要。但这还只是“信息搬过来了”，并不代表模型已经完成了抽象和组合。
FFN 的作用就是：
* 对这个位置当前的表示做更复杂的非线性变换
* 把零散的上下文线索压缩成更高阶特征
### 3.4.1 数学逻辑
$$
\text{SwiGLU}(x) = \left( \text{SiLU}(x W_1) \odot x W_3 \right) W_2
$$

#### 3.4.1.1 门框分支
$$
g = \text{SiLU}(x W_1)
$$
- 先对输入做线性变换：$x \in \mathbb{R}^{d_{\text{model}}},\ W_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$
- 再经过 SiLU 激活，输出一个**软门控权重**
- 作用：**控制信息流**，决定哪些维度保留、哪些抑制

#### 3.4.1.2 信号分支
$$
s = x W_3
$$
- 同样从 $d_{\text{model}}$ 映射到 $d_{\text{ff}}$
- 不经过激活，直接输出**待传输的特征内容**

#### 3.4.1.3 融合与回投影
先做**逐元素相乘**（门控 × 信号）：
$$
g \odot s = \text{SiLU}(x W_1) \odot x W_3
$$

再通过 $W_2$ 投影回原维度：
$$
\text{SwiGLU}(x) = (g \odot s) W_2
$$

### 3.4.2 经典FFN
```
Linear(d_model -> d_ff)
-> ReLU / GELU
-> Linear(d_ff -> d_model)
```
SiLU（也叫 Swish）定义：
$$
\text{SiLU}(x) = x \cdot \sigma(x)
$$
其中 $\sigma(x) = \frac{1}{1+e^{-x}}$ 是 Sigmoid。

对比 ReLU：
- ReLU：$\text{ReLU}(x)=\max(0,x)$，**负轴直接硬截断**，0 点不可导
- SiLU：**全程平滑可导**，负轴缓慢衰减，不会直接杀死梯度

好处：
- 深层网络梯度流动更稳定
- 表达能力更细腻
- 训练更容易收敛、更稳定

所以现代大模型（Llama、Qwen、GPT 系列）几乎全用 SiLU 系激活，不再依赖 ReLU。
> 为什么LLM不约而同用SiLU，而非LeakyReLU等ReLU改造版？
> **核心结论**：LeakyReLU/PReLU等仅解决了ReLU「负半轴死梯度」的皮毛问题，但完全没命中大语言模型对激活函数的核心需求；LLM集体选择SiLU，是**深层训练稳定性、门控表达能力、工程友好性**三个维度的全面碾压。
> 
> 先明确两个激活函数的定义与导数特性：
> - **LeakyReLU**（分段线性，0点不可导）：
>   $$
>   \text{LeakyReLU}(x) = \begin{cases}
>   x, & x>0 \\
>   \alpha x, & x \le 0
>   \end{cases}
>   $$
>   导数是**阶跃跳变**的分段常数：正半轴导数恒为1，负半轴导数恒为$\alpha$（通常取0.01），在$x=0$处导数不连续，存在硬拐点。
> 
> - **SiLU**（全程光滑，处处无限可导）：
>   $$
>   \text{SiLU}(x) = x \cdot \sigma(x), \quad \sigma(x)=\frac{1}{1+e^{-x}}
>   $$
>   导数是连续光滑的自适应函数：
>   $$
>   \text{SiLU}'(x) = \sigma(x) + x \cdot \sigma(x)(1-\sigma(x)) = \sigma(x) \cdot \left[1 + x(1-\sigma(x))\right]
>   $$
>   全定义域无拐点、无跳变，导数随输入平滑变化。
> 
> LLM是**几十上百层的深层Transformer结构**，梯度需要反向传播上百层。
> - LeakyReLU在$x=0$处的导数跳变，会在深层传播中被指数级放大，导致梯度方差剧烈波动，训练极易不稳定、发散；
> - 哪怕$\alpha$调大（比如0.1），也只是缓解了负半轴死梯度，0点的导数不连续问题依然存在，同时负半轴固定斜率会把大量无效噪声的梯度传回网络，干扰特征学习；
> - SiLU的光滑导数特性，让梯度在深层传播中始终保持稳定，极大降低了大模型训练的崩溃风险，这是ReLU系列所有改造版都无法解决的本质缺陷。
> 
> $\sigma(x)$是一个动态的软门控，输入值越大，门控越开放，特征被放行的比例越高；输入值越小，门控越关闭，特征被抑制。这种**自适应门控能力**，和门控FFN的设计思想完全同频，两者结合能实现更精细的信息流控制。
> - LeakyReLU只是一个分段线性映射，**没有任何门控能力**，无法对特征做动态筛选，用在门控FFN中会完全丧失门控的核心优势，表达能力远弱于SiLU。
> 
> LLM的激活值分布有极强的动态性：不同token、不同层、不同训练阶段，激活值的范围和分布差异极大。
> - LeakyReLU的负半轴斜率$\alpha$是固定超参，一旦设定就无法改变，根本无法适配这种动态分布：$\alpha$太小，负半轴梯度几乎为0，和ReLU一样会出现梯度消失；$\alpha$太大，又会引入大量无效梯度，污染特征学习。
> - SiLU的导数是**输入自适应**的：正半轴输入越大，导数越趋近于1，避免梯度饱和；负半轴输入越小，导数越平滑趋近于0，自动过滤无效信息，无需任何超参调整，就能完美适配LLM的动态激活分布。
> 
> 大模型训练的超参调优成本极高：一次7B模型的预训练就要消耗数万卡时，根本没有空间给激活函数的超参做AB测试。
> - LeakyReLU/PReLU都有需要人工调整的超参（$\alpha$），甚至PReLU的$\alpha$是可学习参数，会额外增加模型参数量和训练复杂度；
> - SiLU**没有任何超参数**，开箱即用，无需任何调优，就能在绝大多数场景下取得最优效果，完美契合大模型的工程落地需求。
### 3.4.3 8/3 倍升维
这是**参数量等价**推导出来的经典结论。
- 传统 FFN（老式前馈）
结构：
$$
d_{\text{model}} \to 4d_{\text{model}} \to d_{\text{model}}
$$
> 传统Transformer FFN的行业标准设计就是$\boldsymbol{d_{ff}=4 \times d_{model}}$，这个规范从《Attention Is All You Need》原始论文沿用至今
> $d_{model}=512$，$d_{ff}=2048$，正好是**4倍关系**。后续的BERT、GPT-1/2、T5等所有早期Transformer模型，全部沿用了这个4倍设计
两层矩阵参数量：
$$
d_{\text{model}} \cdot 4d_{\text{model}} + 4d_{\text{model}} \cdot d_{\text{model}}
= 8 d_{\text{model}}^2
$$

- SwiGLU 结构
有三个矩阵：
- $W_1: d_{\text{model}} \to d_{\text{ff}}$
- $W_3: d_{\text{model}} \to d_{\text{ff}}$
- $W_2: d_{\text{ff}} \to d_{\text{model}}$

总参数量：
$$
d_{\text{model}}d_{\text{ff}} + d_{\text{model}}d_{\text{ff}} + d_{\text{ff}}d_{\text{model}}
= 3 d_{\text{model}} d_{\text{ff}}
$$

- 令参数量大致相等
$$
3 d_{\text{model}} d_{\text{ff}} \approx 8 d_{\text{model}}^2
$$

约掉 $d_{\text{model}}$：
$$
d_{\text{ff}} \approx \frac{8}{3} d_{\text{model}}
$$

> $\frac{8}{3}d_{model}$通常不是整数，而GPU的Tensor Core硬件加速，要求维度必须是64/128/256的倍数，因此工业界会把计算结果**向上取整到最近的128的倍数**。
> Llama-7B的$d_{model}=4096$，$\frac{8}{3} \times 4096 \approx 10922.67$，向上取整到128的倍数，最终$d_{ff}=11008$；
> 8/3倍升维不是SwiGLU的强制要求，只是「和传统4倍FFN参数量持平」的标准值。如果你愿意增加FFN的参数量来提升模型效果，完全可以把$d_{ff}$设得更大（比如Llama 3 8B的$d_{ff}=14336$，约为$d_{model}=4096$的3.5倍），只是会增加整体模型的参数量。



### 3.4.4 实现思路

In [ ]:
def silu(x: torch.Tensor) -> torch.Tensor:
    return x * torch.sigmoid(x)
    # sigmoid表达式是1/(1+e^-x)

class SwiGLU(nn.Module):
    def __init__(self, d_model:int, d_ff:int, device=None, dtype=None):
        super().__init__()
        self.w1 = TransformerLinear(d_model, d_ff, device=device, dtype=dtype)
        self.w2 = TransformerLinear(d_ff, d_model, device=device, dtype=dtype)
        self.w3 = TransformerLinear(d_model, d_ff, device=device, dtype=dtype)

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        gate = silu(self.w1(x))
        signal = self.w3(x)
        return self.w2(gate * signal)


## 3.5 RMSNorm
在Transformer架构中，**无论是LayerNorm还是RMSNorm，归一化的对象既不是整个Batch，也不是整个句子序列，而是「每个token自己的特征向量」**。

我们先明确输入张量的维度定义：
$$
x \in \mathbb{R}^{[B, S, D]}
$$
- $B$：Batch Size，批次大小，一次输入的句子数量
- $S$：Sequence Length，序列长度，每个句子的token数量
- $D$：Hidden Dimension / d_model，每个token的特征向量维度

归一化操作**仅在最后一维 $D$ 上独立执行**，也就是说：
- 整个张量里一共有 $B \times S$ 个独立的token特征向量
- 每个向量都会单独做一次归一化，和其他token、其他句子完全无关

这个特性直接决定了：**LayerNorm/RMSNorm和BatchNorm本质上不是一类东西**。BatchNorm是跨Batch在特征维度做归一化，会受批次大小、样本分布的严重影响；而Transformer的Norm层完全独立于批次和序列，天生适配变长文本场景，这也是Transformer完全弃用BatchNorm的核心原因。
### 3.5.1 归一化的意义
#### 3.5.1.1 抑制梯度消失/爆炸
深层Transformer的核心痛点，是**数值在层间的指数级传播**：
- 没有Norm层时，每一层的线性变换、注意力计算，都会让特征的数值尺度发生微小偏移；
- 经过几十上百层的decoder堆叠后，微小的偏移会被指数级放大，最终出现激活值爆炸（数值溢出）、激活值消失（全部趋近于0）；
- 激活值失控会直接导致梯度完全崩掉，模型要么直接不收敛，要么训练中途直接崩溃。

Norm层通过强制约束每个token特征的数值尺度，从根源上解决了这个问题，让深层网络的数值系统始终处于可控范围。
#### 3.5.1.2 平滑优化地形
如果特征的不同维度数值尺度差异极大（比如某一维范围是0~1000，另一维是0~0.001），模型的损失曲面会变成「又瘦又长的峡谷地形」。
此时梯度下降会在峡谷壁上来回横跳，收敛速度极慢，甚至无法收敛到最优解。

Norm层通过归一化，让所有特征维度的尺度保持一致，把崎岖的损失曲面拉成更圆润的「碗状地形」，优化器可以更稳定、更快地朝着最优解前进。
> 这里没太懂说实话，不过没关系下一章就是训练了
### 3.5.2 LayerNorm
LayerNorm是Transformer原始论文就采用的标准归一化方案，完整的数学定义如下：

$$
\text{LayerNorm}(x) = \frac{x - \mu}{\sqrt{\text{Var}(x) + \epsilon}} \cdot \gamma + \beta
$$

其中各个符号的含义：
- $\mu = \frac{1}{d}\sum_{i=1}^d x_i$：当前token特征向量的均值
- $\text{Var}(x) = \frac{1}{d}\sum_{i=1}^d (x_i - \mu)^2$：当前token特征向量的方差
- $\epsilon$：极小值常量（通常取$10^{-5}$），防止分母为0导致的除零错误
- $\gamma$：可学习的缩放权重参数，形状和特征维度一致
- $\beta$：可学习的偏置参数，形状和特征维度一致
它完整做了三件事，每一步都有明确的目的：
1.  **中心化**：$x - \mu$，把特征向量的均值强制拉到0，消除特征的整体偏移
2.  **标准化**：$\frac{x - \mu}{\sqrt{\text{Var}(x) + \epsilon}}$，把特征的标准差强制拉到1，统一所有维度的数值尺度
3.  **仿射变换**：乘以$\gamma$加上$\beta$，给模型保留可学习的调整空间，恢复被归一化丢失的特征表达能力

这个设计非常经典且有效，但也有一个明确的短板：**计算开销偏高**。它既要算均值、又要算方差，还要做减法、除法、仿射变换，在参数量动辄百亿千亿的大模型里，每一点额外的计算开销都会被无限放大。

### 3.5.3 RMSNorm
它的核心论文观点非常直接：**LayerNorm里的「均值中心化」操作，对最终效果的贡献微乎其微，却额外增加了大量计算成本**。

于是RMSNorm直接舍弃了中心化步骤，只保留对特征幅值的约束，用均方根（RMS）做归一化，最终在效果和LayerNorm持平的前提下，大幅降低了计算延迟（论文实测推理速度提升7%~64%）。
$$
\text{RMSNorm}(x_i) = \frac{x_i}{\text{RMS}(x)} \cdot g_i
$$

其中均方根RMS的计算：
$$
\text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{j=1}^d x_j^2 + \epsilon}
$$

符号含义：
- $d$：特征向量的维度
- $\text{RMS}(x)$：当前token特征向量的均方根，衡量特征的整体幅值
- $\epsilon$：防止除零的极小值，和LayerNorm一致
- $g_i$：可学习的缩放权重参数，和特征维度一致

核心优势：
1.  **计算量大幅降低**：只需要计算1次统计量（RMS），省去了均值计算、中心化减法的开销，内存访问更少，算子融合更简单，GPU执行效率更高
2.  **梯度路径更短**：计算图更简单，反向传播时梯度流动更顺畅，深层大模型的训练稳定性更优
3.  **效果完全持平**：Transformer的激活分布本身均值就接近0，中心化操作的收益几乎可以忽略，RMSNorm在各类NLP任务上的效果和LayerNorm无显著差异
4.  **参数量更少**：仅保留缩放权重，舍弃了偏置参数，参数量直接减半，对超大模型来说是可观的显存节省

| 对比维度 | LayerNorm | RMSNorm |
|----------|-----------|---------|
| 核心归一化公式 | $\frac{x - \mu}{\sqrt{\text{Var}(x) + \epsilon}}$ | $\frac{x}{\sqrt{\frac{1}{d}\sum_{j=1}^d x_j^2 + \epsilon}}$ |
| 核心执行逻辑 | 中心化（减均值）+ 标准化（除以标准差） | 仅幅值缩放（除以均方根RMS） |
| 均值中心化操作 | 是，强制将特征均值拉至0 | 否，完全舍弃中心化步骤 |
| 可学习参数 | 缩放权重$\gamma$ + 偏置$\beta$（2组，共$2d$个参数） | 仅缩放权重$g$（1组，共$d$个参数） |
| 核心统计量计算 | 需计算均值、方差2次统计量 | 仅需计算RMS 1次统计量 |
| 计算与推理开销 | 更高：多一次减法广播，内存访问次数更多，算子复杂度更高 | 更低：论文实测推理速度提升7%~64%，大模型场景收益更显著 |
| 显存占用 | 更高：需缓存均值、方差2个中间张量 | 更低：仅需缓存RMS 1个中间张量 |
| 梯度流动特性 | 梯度路径更长，中心化操作引入额外梯度依赖 | 梯度路径更短，计算图更简洁，深层网络梯度稳定性更优 |
| 数值稳定性 | 优秀，对带偏移的特征分布适配性更强 | 同等优秀，Transformer激活分布均值接近0，无中心化适配问题 |
| 工业界主流使用场景 | 早期Transformer（BERT、GPT-1/2）、Encoder-Decoder架构 | 现代Decoder-only大模型（Llama全系列、Qwen2/3、Gemma、Mistral），已成为行业默认标准 |
| 典型结构搭配 | 早期Post-Norm结构为主 | 现代Pre-Norm结构标配 |

### 3.5.4 Pre-Norm
Qwen3、Llama、Gemma等所有现代大模型，都会明确标注「RMSNorm with Pre-Normalization」，这是和原始Transformer最大的结构差异之一。
我们先明确两种结构的执行流程，以单个Decoder层为例：
- **Post-Norm（原始Transformer）**：先做子层计算，再归一化，残差连接在归一化之前
  ```
  x -> Attention -> 残差Add -> LayerNorm -> FFN -> 残差Add -> LayerNorm
  ```
- **Pre-Norm（现代LLM通用）**：先做归一化，再做子层计算，残差连接在子层之后
  ```
  x -> RMSNorm -> Attention -> 残差Add -> RMSNorm -> FFN -> 残差Add
  ```
Pre-Norm结构中，主分支的残差连接是完全直通的，反向传播时梯度可以直接从最后一层流回第一层，不会被Norm层和子层计算阻挡；而Post-Norm的梯度必须经过Norm层，深层网络中极易出现梯度消失。

这个特性让Pre-Norm结构可以轻松堆叠上百层Decoder，而Post-Norm结构在超过30层后，训练稳定性就会急剧下降。

今天主流的Decoder-only大模型，几乎都是**Pre-Norm + RMSNorm + 残差连接**的标准组合，这也是Llama、Qwen等模型的核心架构基底。

### 3.5.5 实现思路

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, d_model:int, eps:float=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.float()  # 计算时使用float32以提高数值稳定性

        mean = x.mean(dim=-1, keepdim=True) # dim=-1表示对最后一个维度求均值，keepdim=True表示保持维度不变，输出的形状与输入相同
        var = x.var(dim=-1, keepdim=True, unbiased=False) # var是方差，unbiased=False表示使用N而不是N-1来计算方差

        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        out = x_norm * self.weight + self.bias
        return out.to(in_dtype)  # 输出与输入保持相同的数据类型
    
class RMSNorm(nn.Module):
    def __init__(self, d_model:int, eps:float=1e-5, device=None, dtype=None):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model, device=device, dtype=dtype))    # 可学习的缩放参数，初始值为1

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.float()

        mean_square = x.pow(2).mean(dim=-1, keepdim=True)   # pow(2)表示每个元素平方，mean表示对最后一个维度求均值
        root_mean_square = torch.sqrt(mean_square + self.eps)  # 加上eps以避免除以零

        out = (x/root_mean_square) * self.weight
        return out.to(in_dtype)